In [2]:
from src.utils.gembase_utils import GembaseUtils
from src.runtime_constants import rr
from src.db import DbConnection

def read_df(guid: str):
    conn = DbConnection(rr.ENV_PROD, True)
    row = conn.select_one("""
    SELECT d.data
      FROM external_data.dms d
     WHERE d.guid = %s
    """, [guid])

    conn.close()
    df = GembaseUtils.db_data_to_df(row["data"])
    return df

df = read_df("7de9c732-cb01-421c-b2ac-c77a88d9f965")
df

,index,base,gender_female,age,behaviors__chill,behaviors__collect,behaviors__combat,behaviors__compete,behaviors__complete,behaviors__cooperate,...,topics__action,topics__adventure,topics__comedy,topics__drama,topics__fantasy,topics__horror,topics__mystery,topics__realistic,topics__romance,topics__sci_fi
0,core__4x,-0.286876,-0.236222,-0.279012,-0.544473,-0.437139,0.945730,-0.060297,-0.039705,-0.006109,...,-0.125153,0.271123,-1.201997,0.152743,0.935394,0.357272,0.357049,-0.124540,0.693785,0.567681
1,core__agriculture,-1.613746,-2.391151,3.101368,1.931769,-1.539575,-1.007248,-1.145862,6.340319,-0.514554,...,-0.811611,-1.817877,-1.855809,-1.705711,-0.443348,-0.290245,2.885535,0.201600,1.876005,1.601042
2,core__artillery,0.185683,0.345047,3.976050,1.740851,-3.402470,-2.010398,0.155788,-2.015860,-0.089300,...,2.663766,-0.182689,-0.027521,-1.593802,5.034263,-1.107471,-3.677847,1.793501,3.040210,1.054779
3,core__asymetric,0.186699,0.541895,-1.925720,0.206369,1.681497,2.365393,2.511573,-1.760296,-1.168963,...,-2.387840,-1.420337,-2.283225,1.538963,-0.206982,1.533624,1.822212,2.505516,-0.735267,0.782608
4,core__auto_chess,-0.324254,-0.540412,0.898060,0.088691,-0.062033,-0.381284,-0.206233,-1.022817,0.121688,...,0.278240,-0.090367,-0.933755,0.583697,0.517207,-0.089260,-0.412431,0.926945,0.739697,0.058781
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,roles__rebel,0.680609,0.218046,0.714625,0.191649,0.356736,0.347430,0.401425,0.186171,0.594046,...,-0.073013,0.392747,0.082462,-0.440938,0.166528,0.870036,0.395634,0.198932,0.530709,0.843577
187,roles__solver,1.009763,-0.057748,0.149161,0.558327,-0.250703,0.455224,-0.090365,-1.619912,0.865482,...,-0.898146,0.017233,-0.326589,-0.834499,0.581877,-0.108205,1.472283,0.524016,0.174241,1.513632
188,roles__team,0.101908,-0.249952,1.301594,0.831747,1.302855,-0.510963,0.635418,-0.371966,1.065682,...,0.453690,0.917116,-0.015566,0.458056,0.829150,0.737618,-0.120760,0.397845,-0.891099,0.591549
189,roles__villain,-1.224167,-0.661129,0.239529,0.693033,-0.133359,0.587165,0.539171,-1.178606,0.770367,...,-0.077383,0.840105,-0.406304,0.641862,0.947944,1.305729,-0.233411,0.322599,0.393001,1.141359


In [8]:
df = read_df("057c206c-42b5-4f6d-915d-4f772f13e761")
df["core__4x"].min()

-4.903996025301266

In [9]:
df = read_df("057c206c-42b5-4f6d-915d-4f772f13e761")
df["core__4x"].max()

5.875363536736288

In [35]:
df = read_df("057c206c-42b5-4f6d-915d-4f772f13e761")


In [36]:
from scipy import stats
import numpy as np

for col in df.columns:
    if col == "survey_instance":
        continue
    df_tmp = df[col][np.abs(stats.zscore(df[col])) < 3]
    #print(df_tmp.to_numpy())
    min_val = np.min(df_tmp.to_numpy())
    max_val = np.max(df_tmp.to_numpy())
    normalized_col = "n___" + col
    df[normalized_col] = df.apply(lambda r: (r[col] - min_val) / (max_val - min_val), axis=1)
    df[normalized_col].clip(lower=0, upper=1, inplace=True)
    break

df



,core__4x,core__agriculture,core__artillery,core__asymetric,core__auto_chess,core__automation,core__backtracking,core__ball,core__baseball,core__basketball,...,roles__explorer,roles__follower,roles__mystic,roles__rebel,roles__solver,roles__team,roles__villain,roles__warrior,survey_instance,n___core__4x
0,2.578229,-1.414655,0.364878,0.440739,0.614512,0.360691,1.907857,1.854493,10.925504,-13.549215,...,1.931276,1.647940,2.196893,1.729600,1.234919,0.975751,1.180275,1.375167,33cbe449-af6a-484b-a521-6a4d08f73454,0.630131
1,2.790493,4.551493,1.228174,4.850972,4.109786,-1.565285,4.689094,2.490101,0.695541,17.479627,...,3.922272,1.978660,3.601308,2.797976,2.391544,2.111298,2.791318,2.404181,5db16995-594b-4b4e-9151-e502b85d6ff3,0.653943
2,-0.845301,-4.483546,7.304679,-4.284276,1.010088,5.375650,-2.523223,1.308408,7.330665,-9.524752,...,-1.308801,0.400054,0.628115,-1.972387,-0.359957,0.559632,0.407841,0.602326,9722c487-9a13-4a9d-8599-577b69805034,0.246084
3,3.683048,9.791565,3.609840,2.451747,2.739379,4.569710,4.435884,3.277891,-14.994048,21.062342,...,3.248923,2.896846,4.011653,3.828898,3.582520,2.787527,3.154025,4.587344,7094ec2f-88d4-4642-90f6-f12b41eca4f1,0.754069
4,2.873382,2.293441,7.531144,4.993248,1.819800,-2.033752,4.030513,4.650238,3.701675,10.187288,...,3.872421,2.229041,2.536022,2.283093,1.966257,2.720388,2.171839,3.301000,eb6a4d45-c680-4eaf-bef7-aebe5edb7ce0,0.663241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1965,4.900399,5.288770,2.834282,2.145343,3.573811,4.291212,2.934931,3.352512,-15.904767,-10.270780,...,3.023228,2.169048,3.186277,2.607685,2.512467,2.385055,2.771769,2.533438,35fc0bbf-cef7-4ad9-b0c7-8bfb50043cc3,0.890630
1966,2.217039,4.326619,-9.516410,-0.042601,0.018577,-3.631029,3.210302,0.879101,-6.320245,0.469428,...,-0.326479,-1.354748,-1.594075,-0.372191,0.536151,0.215736,0.887133,0.349996,b11b6d9c-f739-478a-ab26-d26d11825827,0.589614
1967,3.418625,5.155607,-4.228115,0.234306,0.500793,-0.049113,1.065855,2.218347,-8.549056,3.914636,...,2.682185,2.075568,1.630780,2.213591,3.576500,1.866416,2.341371,3.258449,e9c3e858-826f-479e-a85d-6ea3b0e4e43a,0.724406
1968,1.783681,-0.742899,1.692248,-5.018639,2.428023,4.402909,3.027022,1.274490,-15.067321,10.905596,...,1.502451,0.865052,1.856688,2.023850,0.354875,2.742764,1.808641,3.704205,b6e5ec83-0b04-4f2d-8d54-1bc0d6785adc,0.541000


In [41]:
data = []
for index, row in df.iterrows():
    for col in df.columns:
        if "n__" in col:
            val = row[col]
            val = int(val * 100)
            if val >= 80:
                val = 100
            elif val <= 20:
                val = 0
            else:
                val = 50
            print(col, val)
            data.append((row["survey_instance"], col.replace("n___", ""), val))

data

n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 50
n___core__4x 100
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 50
n___core__4x 0
n___core__4x 50
n___core__4x 50


[('33cbe449-af6a-484b-a521-6a4d08f73454', 'core__4x', 50),
 ('5db16995-594b-4b4e-9151-e502b85d6ff3', 'core__4x', 50),
 ('9722c487-9a13-4a9d-8599-577b69805034', 'core__4x', 50),
 ('7094ec2f-88d4-4642-90f6-f12b41eca4f1', 'core__4x', 50),
 ('eb6a4d45-c680-4eaf-bef7-aebe5edb7ce0', 'core__4x', 50),
 ('468822fe-53f1-4cd6-8d35-9c28da8bf35d', 'core__4x', 50),
 ('29f5cee1-66ef-4df7-b9e5-15c6064af67f', 'core__4x', 50),
 ('cc5a5c9f-65f2-449a-9e51-a1cc842c652d', 'core__4x', 50),
 ('91e280cd-23dc-4a2f-8f01-8a660ec45863', 'core__4x', 50),
 ('761263df-a83b-4499-9acd-3f7cc3fac193', 'core__4x', 50),
 ('e7a43a08-289e-440b-acfd-793b3de989c8', 'core__4x', 0),
 ('612d9fba-1550-4777-8f78-1d05de18e000', 'core__4x', 50),
 ('7dcb3d1f-351d-4471-b98c-b0df3d079de2', 'core__4x', 50),
 ('74cb352e-1319-4d45-acff-8dcf358f0a0f', 'core__4x', 50),
 ('498b4f54-fdea-45df-8789-7aadc51dd6f5', 'core__4x', 50),
 ('4b20bac9-de87-4047-9393-33e167c2d4d9', 'core__4x', 50),
 ('346f185f-e627-47c0-aca3-74ba7ea1ecef', 'core__4x', 50)